# Lab 3 · NumPy trên dữ liệu giá và ma trận review

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành**  
**BẢN PILOT — dùng thử private grader; chưa phải điểm chính thức của môn học.**

Giữ bài làm trong notebook này. Với Colab: **File → Save a copy in Drive** trước khi sửa.
Khi chuyển sang GitHub, nộp đúng file `lab-03.ipynb`; lưu trên Drive chưa phải nộp bài.

## Mục tiêu

1. Đưa giá vào NumPy, nhận biết và xử lý NaN chủ động.
2. Đếm/lọc bằng mặt nạ bool và ghép điều kiện tại ngưỡng.
3. Dùng `np.where`, phân vị và `argsort`.
4. Dùng đúng `axis`, `argmax` và broadcasting trên ma trận.

## Cách làm việc

- Khởi động và bài có hướng dẫn: tự gõ, **không dùng AI**, theo chính sách môn học.
- Phần **Bài tự làm ✅ mở**: được dùng AI, phải ghi prompt, điều đã kiểm chứng và giới hạn.
- Điền cell **Bài làm** và **Trả lời**. Giữ tên hàm/biến, mã câu và các cell công khai đã cho.
- Public checks chỉ cho biết ví dụ nhỏ đã qua. Grader sẽ dùng dữ liệu khác cùng yêu cầu công khai.
- Cell chưa làm được báo `CHƯA LÀM`; một public check lỗi không dừng các check khác.
  Cơ chế này không che lỗi cú pháp ở cell bài làm: bạn vẫn cần sửa lỗi cú pháp để Run all được.
- **Restart & Run all** trước khi nộp. Việc notebook chạy hết với TODO chưa hoàn thành không có nghĩa bài đã đạt.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Q1: làm sạch NaN | 10 | Tự động |
| Q2: mặt nạ và tỷ lệ | 15 | Tự động |
| Q3: nhãn theo trung vị | 10 | Tự động |
| Q4: phân vị và ngoại lai | 10 | Tự động |
| Q5: tổng hợp ma trận | 15 | Tự động |
| Q6: vị trí cực đại | 10 | Tự động |
| Q7: top-k bằng argsort | 10 | Tự động |
| Giải thích, kiểm chứng, khai báo AI | 20 | Giảng viên |

Q1–Q6 dùng phép toán NumPy vector hóa; không dùng pandas, vòng lặp Python hoặc list comprehension
để tính kết quả. Q7 thuộc phần mở, phải dùng `np.argsort`. Không sửa mảng đầu vào tại chỗ.
Các yêu cầu phương pháp này được giảng viên kiểm tra cùng code, không suy ra chỉ từ test kết quả.

Điểm được tính theo từng mục, không theo số lượng public checks. Phần diễn giải do giảng viên đọc.
Các câu độc lập dự kiến được chấm với đầu vào riêng; không trừ toàn bộ lab vì một hàm trước đó sai.


In [1]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")


In [2]:
import numpy as np


## Phần 0 · Khởi động có hướng dẫn — khoảng 10 phút

Tự gõ và thay dữ liệu để quan sát. Không chấm cứng dtype là `int64`: kiểu số nguyên mặc định
có thể phụ thuộc môi trường. Điều quan trọng ở đây là kích thước và nhóm kiểu dữ liệu.


In [3]:
a = np.array([4, 8, 15, 16, 23, 42])
print("shape / size / dtype:", a.shape, a.size, a.dtype)
print("Có kiểu số nguyên:", np.issubdtype(a.dtype, np.integer))
diem = np.array([7.5, 4.0, 9.0, 5.5])
print("Mặt nạ đạt:", diem >= 5, "Số đạt:", (diem >= 5).sum())
M_demo = np.array([[1, 2, 3], [4, 5, 6]])
print("Cộng theo cột (axis=0):", M_demo.sum(axis=0))
print("Cộng theo hàng (axis=1):", M_demo.sum(axis=1))


shape / size / dtype: (6,) 6 int64
Có kiểu số nguyên: True
Mặt nạ đạt: [ True False  True  True] Số đạt: 3
Cộng theo cột (axis=0): [5 7 9]
Cộng theo hàng (axis=1): [ 6 15]


## Dữ liệu: thử nhanh và thực hành trên snapshot thật

**Bản nháp mở sẵn bằng 6 dòng giả lập**, chứa giá thiếu và giá bằng 0 để thử quy tắc xử lý.
Không dùng kết quả từ 6 dòng này để kết luận về Santiago.

Khi thực hành trên dữ liệu thật, lấy `listings.csv` bản **visualisations**, Santiago,
snapshot **2026-06-29**, giá **CLP/đêm**, từ [Inside Airbnb](https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations/listings.csv),
hoặc bản sao cùng snapshot do giảng viên cung cấp. Đưa file vào Colab/Jupyter rồi điền
`LISTINGS_PATH` trong cell dưới. Nếu nguồn tải không hoạt động, không thay bằng snapshot khác
mà không ghi rõ. Đường dẫn là cấu hình dữ liệu thực hành, không phải câu trả lời.

Public checks dùng các bộ dữ liệu nhỏ riêng, nên vẫn chạy được khi chưa có snapshot thật.
Các con số của snapshot gốc không được dùng làm hằng số trong lời giải.


In [5]:
from pathlib import Path
import csv
import io

# Nếu đã có snapshot thật, điền đường dẫn, ví dụ "data/listings.csv".
# Để None khi xem bản nháp: dùng 6 dòng GIẢ LẬP ở dưới, không phải Santiago.
LISTINGS_PATH = "dse2049_1_25023073/listings.csv"
DEMO_CSV = 'id,name,neighbourhood,room_type,price,number_of_reviews,last_review\n1,Phong A,Centro,Entire home/apt,100,0,\n2,Phong B,Centro,Private room,50,2,2026-05-10\n3,Phong C,Norte,Entire home/apt,,0,\n4,Phong D,Norte,Entire home/apt,150,1,2026-06-01\n5,Phong E,Sur,Private room,0,3,2026-05-30\n6,Phong F,Sur,Entire home/apt,300,2,2026-06-10\n'

if LISTINGS_PATH is None:
    rows = list(csv.DictReader(io.StringIO(DEMO_CSV)))
    DATA_LABEL = "GIẢ LẬP — 6 dòng để thử bản nháp"
else:
    with open(LISTINGS_PATH, encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))
    DATA_LABEL = f"File người học cung cấp: {LISTINGS_PATH}"

print(DATA_LABEL)
print("Số dòng:", len(rows))
print(rows[:2])


File người học cung cấp: dse2049_1_25023073/listings.csv
Số dòng: 18534
[{'id': '978070332077815549', 'name': 'luminosa mansarda con balcón', 'host_id': '118157228', 'host_profile_id': '1468207531264648757', 'host_name': 'Patricia', 'neighbourhood_group': '', 'neighbourhood': 'Ñuñoa', 'latitude': '-33.43765', 'longitude': '-70.5833', 'room_type': 'Private room', 'price': '45647', 'minimum_nights': '4', 'number_of_reviews': '2', 'last_review': '2023-12-15', 'reviews_per_month': '0.06', 'calculated_host_listings_count': '5', 'availability_365': '269', 'number_of_reviews_ltm': '0', 'license': ''}, {'id': '1069858035768058539', 'name': 'Cómoda habitación bien ubicada con baño privado', 'host_id': '462786436', 'host_profile_id': '1470084981980135796', 'host_name': 'Luís Alfonso', 'neighbourhood_group': '', 'neighbourhood': 'Recoleta', 'latitude': '-33.42242', 'longitude': '-70.64116', 'room_type': 'Private room', 'price': '19856', 'minimum_nights': '1', 'number_of_reviews': '8', 'last_revie

In [ ]:
# Chỉ dùng pandas ở bước đọc/chuyển cột giá; các câu tính toán bên dưới dùng NumPy.
import pandas as pd
gia = pd.to_numeric(pd.DataFrame(rows)["price"], errors="coerce").to_numpy(dtype=float)
print("Mảng giá:", gia[:8], "shape:", gia.shape)


Mảng giá: [100.  50.  nan 150.   0. 300.] shape: (6,)


## Phần 1 · Xử lý giá bằng NumPy — khoảng 40 phút

Các hàm nhận mảng qua tham số. Bộ chấm có thể thay giá, độ dài và vị trí NaN; không sử dụng
số phòng, số NaN hay median Santiago như hằng số. Trong phạm vi lab này, giá không âm;
chỉ Q1 nhận NaN, các câu khác nhận dữ liệu sạch theo mô tả.


### Q1 · Tách NaN khỏi giá hợp lệ — 10 điểm

`clean_prices(values)` nhận ndarray float 1 chiều, có thể rỗng hoặc toàn NaN,
không có Infinity. Trả tuple `(clean, n_missing)`:
`clean` là ndarray 1 chiều giữ thứ tự và giữ giá 0; `n_missing` là số NaN.
Dùng `np.isnan` và mặt nạ, không thay NaN bằng 0.


In [ ]:
def clean_prices(values):
    # TODO
    mask = np.isnan(values)
    clean = values[~mask]
    n_missing = np.sum(mask)
    return clean, n_missing


In [ ]:
def check_q1_1():
    values = np.array([10.0, np.nan, 0.0, 20.0])
    clean, missing = clean_prices(values)
    assert isinstance(clean, np.ndarray)
    np.testing.assert_array_equal(clean, np.array([10.0, 0.0, 20.0]))
    assert missing == 1

public_check("Q1 · ví dụ 1", check_q1_1)


[ĐẠT] Q1 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q2 · Tỷ lệ và khoảng giá — 15 điểm

`price_masks(prices, low, high)` nhận ndarray 1 chiều không rỗng, số hữu hạn, và `low < high`.
Trả dict `share_above` = tỷ lệ giá **> high**, `n_below` = số giá **< low**,
`n_between` = số giá trong **[low, high]**, gồm hai biên. Tỷ lệ ở thang 0–1, không làm tròn.
Dùng mặt nạ bool và toán tử `&` để ghép khoảng.


In [ ]:
def price_masks(prices, low, high):
    # TODO
    mask1 = prices < low
    mask2 = prices > high
    mask3 = (prices >= low) & (prices <= high)
    share_above = np.sum(mask2) / len(prices)
    n_below = np.sum(mask1)
    n_between = np.sum(mask3)
    return {
        "share_above": share_above,
        "n_below": n_below,
        "n_between": n_between
}


In [ ]:
def check_q2_1():
    result = price_masks(np.array([0.0, 10.0, 20.0, 30.0]), 10.0, 20.0)
    assert set(result) == {"share_above", "n_below", "n_between"}
    assert np.isclose(result["share_above"], 0.25)
    assert result["n_below"] == 1 and result["n_between"] == 2

public_check("Q2 · ví dụ 1", check_q2_1)


[ĐẠT] Q2 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q3 · Nhãn giá theo trung vị — 10 điểm

`label_prices(prices)` nhận mảng sạch không rỗng. Trả ndarray chuỗi cùng shape:
`"cao"` khi giá **> median**, còn lại `"pho thong"`, kể cả giá đúng median.
Dùng `np.median` và `np.where`. Nhóm cao không bắt buộc chiếm đúng 50%.


In [ ]:
def label_prices(prices):
    # TODO
    median = np.median(prices)
    return np.where(prices > median, "cao", "pho thong")


In [ ]:
def check_q3_1():
    result = label_prices(np.array([10.0, 20.0, 20.0, 30.0]))
    assert isinstance(result, np.ndarray)
    np.testing.assert_array_equal(result, ["pho thong", "pho thong", "pho thong", "cao"])

public_check("Q3 · ví dụ 1", check_q3_1)


[ĐẠT] Q3 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q4 · Phân vị và ngoại lai — 10 điểm

`percentile_outliers(prices, q_low=1, q_high=99)` nhận mảng sạch không rỗng,
`0 <= q_low < q_high <= 100`. Tính `np.percentile(..., method="linear")`.
Trả `(p_low, p_high, n_outside)`; chỉ đếm giá **< p_low hoặc > p_high**, không gồm biên.
Không làm tròn ngưỡng trước khi so sánh; số trùng nhau được tính theo từng phần tử.


In [ ]:
def percentile_outliers(prices, q_low=1, q_high=99):
    # TODO
    p_low = np.percentile(prices, q_low, method ='linear')
    p_high = np.percentile(prices, q_high, method = 'linear')
    mask1 = prices < p_low
    mask2 = prices > p_high
    n_outside = np.sum(mask1) + np.sum(mask2)
    return (p_low, p_high, n_outside)

In [ ]:
def check_q4_1():
    low, high, count = percentile_outliers(np.array([0.0, 10.0, 20.0, 30.0]), 25, 75)
    assert np.isclose(low, 7.5) and np.isclose(high, 22.5)
    assert count == 2

public_check("Q4 · ví dụ 1", check_q4_1)


[ĐẠT] Q4 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Phần 2 · Ma trận review — khoảng 20 phút

Ma trận sau được giữ từ notebook gốc: hàng là năm 2022–2025, cột là tháng 1–12;
số liệu review Santiago được dùng để luyện `axis` và chỉ số, chưa tự chứng minh nguyên nhân biến động.
Public checks dùng ma trận nhỏ khác để kiểm tra cách tính độc lập với các con số này.


In [ ]:
M = np.array([
    [3231,2854,3469,3449,3536,3155,4580,4509,4963,4951,5100,4590],
    [5315,4584,5555,5186,5066,5122,6997,7440,6503,7528,8523,7371],
    [8218,6959,9330,9997,8299,9705,13066,13659,11722,12480,15176,12656],
    [14636,11652,16565,14041,14509,15192,21726,22904,17301,20791,23028,19087],
])
years = np.array([2022, 2023, 2024, 2025])
print(M.shape)


(4, 12)


### Q5 · Tổng theo năm và trung bình theo tháng — 15 điểm

`matrix_summary(matrix)` nhận ndarray 2 chiều số hữu hạn, ít nhất 1 hàng và 1 cột.
Trả tuple `(row_totals, column_means)` là hai ndarray 1 chiều:
tổng từng hàng và trung bình từng cột. Không giả định luôn có 4 hàng, 12 cột.


In [ ]:
def matrix_summary(matrix):
    # TODO
    
    return (np.sum(matrix, axis = 1),np.mean(matrix, axis = 0))


In [ ]:
def check_q5_1():
    totals, means = matrix_summary(np.array([[1.0, 2.0, 3.0], [3.0, 4.0, 5.0]]))
    assert isinstance(totals, np.ndarray) and isinstance(means, np.ndarray)
    np.testing.assert_array_equal(totals, [6.0, 12.0])
    np.testing.assert_allclose(means, [2.0, 3.0, 4.0])

public_check("Q5 · ví dụ 1", check_q5_1)


[ĐẠT] Q5 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Q6 · Vị trí cực đại — 10 điểm

`peak_positions(matrix, row_labels)` nhận ma trận như Q5; `row_labels` là ndarray nhãn năm
số nguyên, dài bằng số hàng. Trả `(peak_columns, first_column_year)`.
`peak_columns`: ndarray vị trí cột lớn nhất của từng hàng, đánh số **từ 1**.
`first_column_year`: nhãn năm có giá trị cột đầu lớn nhất. Đồng hạng lấy vị trí xuất hiện đầu tiên.
Dùng `np.argmax`; không nhầm chỉ số hàng với nhãn năm.


In [ ]:
def peak_positions(matrix, row_labels):
    # TODO
    peak_columns = np.argmax(matrix, axis = 1)+1
    max_col = np.argmax(matrix[:,0])
    first_column_year = row_labels[max_col] 
    return (peak_columns, first_column_year)

In [ ]:
def check_q6_1():
    positions, year = peak_positions(np.array([[1, 5, 2], [4, 2, 3]]), np.array([2022, 2025]))
    assert isinstance(positions, np.ndarray)
    np.testing.assert_array_equal(positions, [2, 1])
    assert year == 2025

public_check("Q6 · ví dụ 1", check_q6_1)


[ĐẠT] Q6 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


## Phần 3 · Bài tự làm ✅ mở

Q7 kế thừa bài “10 giá lớn nhất bằng argsort” và được đưa vào rubric tự động.
Được dùng AI, kèm dự đoán và kiểm chứng. Bài broadcasting và bootstrap bên dưới là mở rộng.


### Q7 · Top-k giá bằng argsort — 10 điểm

`top_prices(prices, k)` nhận ndarray giá sạch 1 chiều, có thể rỗng; `k` là số nguyên không âm.
Trả ndarray tối đa `k` giá lớn nhất, **giảm dần**, giữ các giá trùng theo số lần xuất hiện.
`k=0` trả mảng rỗng; `k` lớn hơn số phần tử trả tất cả giá. Phải dùng `np.argsort`,
không sửa mảng gốc. Thử `k=10` khi thực hành trên Santiago.


In [ ]:
def top_prices(prices, k):
    # TODO
    if k == 0:
        return np.array([], dtype = prices.dtype)
    else:
        des_i = np.argsort(-prices)
        top_k = des_i[:k]
        return prices[top_k]


In [ ]:
def check_q7_1():
    result = top_prices(np.array([10.0, 30.0, 20.0]), 2)
    assert isinstance(result, np.ndarray)
    np.testing.assert_array_equal(result, [30.0, 20.0])

public_check("Q7 · ví dụ 1", check_q7_1)


[ĐẠT] Q7 · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
def check_q7_2():
    result = top_prices(np.array([10.0, 30.0]), 0)
    assert isinstance(result, np.ndarray) and result.shape == (0,)

public_check("Q7 · ví dụ 2", check_q7_2)


[ĐẠT] Q7 · ví dụ 2: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


### Áp dụng các hàm lên dữ liệu thực hành

Giá dùng nguồn đã chọn ở trên; ma trận dùng số review trích trong notebook gốc.
Hai phần chạy riêng để lỗi xử lý giá không chặn phần ma trận.


In [ ]:
def price_report():
    clean, missing = clean_prices(gia)
    if clean.size == 0:
        return {"missing": missing, "note": "Không có giá hợp lệ để phân tích."}
    return {"source": DATA_LABEL, "missing": missing,
            "masks": price_masks(clean, 50000, 100000),
            "labels_preview": label_prices(clean)[:5],
            "outliers": percentile_outliers(clean),
            "top10": top_prices(clean, 10)}

preview("Giá", price_report)
preview("Tổng và trung bình review", lambda: matrix_summary(M))
preview("Vị trí cực đại review", lambda: peak_positions(M, years))


Giá
{'source': 'GIẢ LẬP — 6 dòng để thử bản nháp', 'missing': np.int64(1), 'masks': {'share_above': np.float64(0.0), 'n_below': np.int64(5), 'n_between': np.int64(0)}, 'labels_preview': array(['pho thong', 'pho thong', 'cao', 'pho thong', 'cao'], dtype='<U9'), 'outliers': (np.float64(2.0), np.float64(294.0), np.int64(2)), 'top10': array([300., 150., 100.,  50.,   0.])}
Tổng và trung bình review
(array([ 48387,  75190, 131267, 211432]), array([ 7850.  ,  6512.25,  8729.75,  8168.25,  7852.5 ,  8293.5 ,
       11592.25, 12128.  , 10122.25, 11437.5 , 12956.75, 10926.  ]))
Vị trí cực đại review
(array([11, 11, 11, 11]), np.int64(2025))


### Diễn giải, kiểm chứng và khai báo — 20 điểm

1. Đưa một mảng có giá đúng median; giải thích vì sao nhóm `cao` không nhất thiết chiếm 50% (5 điểm).
   + Ví dụ mảng: np.array([10.0, 20.0, 20.0, 30.0]). Trung vị (median) là 20.0.
   + Theo định nghĩa, nhóm cao là các phần tử có giá strictly lớn hơn median (prices > median).
   + Trong mảng trên, chỉ có duy nhất giá trị 30.0 là lớn hơn 20.0, nên nhóm cao chỉ chiếm 1/4 = 25%, còn 3 phần tử [10.0, 20.0, 20.0] được gắn nhãn pho thong. Khi dữ liệu có nhiều giá trị trùng bằng median, nhóm cao sẽ chiếm ít hơn 50%.
2. Chọn một hàng/cột nhỏ và tính tay để kiểm chứng Q5 hoặc Q6 (5 điểm).
   + Với ma trận $2 \times 3$: $M = \begin{bmatrix} 1 & 2 & 3 \ 3 & 4 & 5 \end{bmatrix}$.
   + Kiểm chứng Q5:
   + Tổng theo hàng (axis=1): Hàng 1 = $1+2+3 = 6$, Hàng 2 = $3+4+5 = 12 \rightarrow$ Kết quả [6.0, 12.0].
   + Trung bình theo cột (axis=0): Cột 1 = $(1+3)/2 = 2$, Cột 2 = $(2+4)/2 = 3$, Cột 3 = $(3+5)/2 = 4 \rightarrow$ Kết quả [2.0, 3.0, 4.0].
   + Khớp hoàn toàn với output của hàm matrix_summary.
3. Vì sao số review thô tăng chưa đủ để khẳng định nguyên nhân hay tách riêng mùa vụ?
   + Số review thô tăng qua các năm có thể do nền tảng mở rộng quy mô (tổng số lượng phòng listing mới tăng lên) chứ không nhất thiết do từng phòng có nhiều khách hơn hoặc do hiệu ứng mùa vụ.
   Nêu một cách chuẩn hóa và một giới hạn còn lại (5 điểm).
   + Chuẩn hóa theo hàng để loại bỏ quy mô chung của năm và nhìn rõ phân bố mùa vụ tương đối giữa các tháng; hoặc chia tổng review cho tổng số listings hoạt động.
4. Khai báo AI cho phần mở; ghi một trường hợp bạn tự thử ngoài public checks,
   hoặc nói rõ không dùng AI và trình bày cách tự kiểm chứng (5 điểm).
   + Đã tự kiểm tra hàm clean_prices với mảng toàn np.nan hoặc mảng rỗng np.array([]), hàm trả về mảng rỗng hợp lệ và đếm đúng số missing.


**Trả lời của tôi:**

[Điền câu trả lời, số liệu hoặc bằng chứng ở đây.]


### E1 · Broadcasting để nhìn mùa vụ

**Hàm:** `normalize_rows(matrix: np.ndarray) -> np.ndarray`.

- Nhận ndarray 2 chiều các số hữu hạn **không âm**, ít nhất một hàng và một cột.
- Trả ndarray float, cùng shape, chia mỗi phần tử cho trung bình của chính hàng đó.
  Tính trung bình với `axis=1, keepdims=True` và dùng broadcasting.
- **Quy tắc hàng có mean bằng 0:** trả toàn số 0 ở hàng đó; không trả NaN/Infinity và tránh chia cho 0.
  Quy tắc được chốt để mọi bài có cùng hợp đồng; bạn vẫn cần diễn giải vì sao hàng đó không có tín hiệu mùa vụ.
- Không sửa mảng đầu vào; không dùng vòng lặp Python/list comprehension.
- Đây là phần mở rộng chưa tính điểm tự động, không thay đổi rubric Q1–Q7.


In [ ]:
def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    """Chia từng hàng cho mean của hàng; hàng mean=0 trả toàn 0.

    Input: ma trận 2D không âm, hữu hạn, không rỗng.
    Output: ndarray float cùng shape; không sửa input.
    """
    matrix = np.asarray(matrix, dtype=float)
    row_means = np.mean(matrix, axis=1, keepdims=True)
    safe_means = np.where(row_means == 0, 1.0, row_means)
    return np.where(row_means == 0, 0.0, matrix / safe_means)


In [ ]:
def check_e1_1():
    matrix = np.array([[1, 2, 3], [2, 4, 6]])
    before = matrix.copy()
    result = normalize_rows(matrix)
    assert isinstance(result, np.ndarray) and result.shape == matrix.shape
    assert np.issubdtype(result.dtype, np.floating)
    np.testing.assert_allclose(result, [[0.5, 1.0, 1.5], [0.5, 1.0, 1.5]])
    np.testing.assert_array_equal(matrix, before)

public_check("E1 · mở rộng · ví dụ 1", check_e1_1)


[ĐẠT] E1 · mở rộng · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
def check_e1_2():
    matrix = np.array([[0.0, 0.0], [2.0, 2.0]])
    with np.errstate(divide="raise", invalid="raise"):
        result = normalize_rows(matrix)
    assert np.isfinite(result).all()
    np.testing.assert_allclose(result, [[0.0, 0.0], [1.0, 1.0]])

public_check("E1 · mở rộng · ví dụ 2", check_e1_2)


[ĐẠT] E1 · mở rộng · ví dụ 2: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
preview("Ma trận review đã chuẩn hóa theo hàng", lambda: normalize_rows(M))


Ma trận review đã chuẩn hóa theo hàng
[[0.8012896  0.70779342 0.86031372 0.85535371 0.87692975 0.78244156
  1.13584227 1.11823424 1.23082646 1.22785046 1.26480253 1.13832228]
 [0.8482511  0.73158665 0.88655406 0.82766325 0.80851177 0.81744913
  1.11669105 1.18739194 1.03785078 1.20143636 1.36023407 1.17637984]
 [0.75126269 0.63616903 0.8529181  0.91389306 0.75866745 0.88719937
  1.194451   1.24866113 1.07158692 1.1408808  1.38734031 1.15697014]
 [0.83067842 0.66131901 0.94016043 0.7969087  0.82347043 0.86223467
  1.2330773  1.29993568 0.98193273 1.18001059 1.3069734  1.08329865]]


### E2 · Bootstrap trung vị — nâng cao, không bắt buộc

**Hàm:** `bootstrap_median(prices: np.ndarray, n_boot: int = 1000, seed: int = 42) -> dict`.

- Input: ndarray float 1 chiều không rỗng, giá hữu hạn không âm, **đã bỏ NaN**;
  `n_boot` là số nguyên dương, `seed` là số nguyên không âm.
- Tạo bộ sinh cục bộ `rng = np.random.default_rng(seed)`. Theo thứ tự từ lần 0 tới `n_boot - 1`,
  mỗi lần gọi `rng.choice(prices, size=prices.size, replace=True)` rồi tính median mẫu.
  Cho phép vòng lặp Python ở bài này; không tạo lại RNG trong từng vòng.
- Trả dict có đúng ba khóa: `medians` (ndarray float 1 chiều dài `n_boot`, thứ tự các lần lấy mẫu),
  `ci_low`, `ci_high` (float; phân vị 2.5 và 97.5 của `medians`, `method="linear"`). Không làm tròn.
- Cùng dữ liệu, `n_boot`, `seed` phải cho cùng kết quả; không sửa input hay trạng thái RNG toàn cục.
  Dữ liệu một phần tử vẫn hợp lệ. Không yêu cầu xử lý input ngoài hợp đồng trên.
- Tách tính toán khỏi in/vẽ. Cell gọi hàm dùng mảng sạch nhỏ để xem nháp; khi thực hành,
  truyền giá Santiago đã làm sạch. Public checks dùng ít vòng để chạy nhanh.

Việc dùng RNG, seed và thứ tự lấy mẫu cụ thể giúp grader tái lập kết quả.
Khoảng bootstrap vẫn cần được diễn giải trong bối cảnh chất lượng và tính đại diện của dữ liệu.


In [ ]:
def bootstrap_median(prices: np.ndarray, n_boot: int = 1000, seed: int = 42) -> dict:
    """Trả {"medians": ndarray float, "ci_low": float, "ci_high": float}.

    Mỗi vòng: rng.choice cùng kích thước input, có hoàn lại, rồi median.
    CI: percentile [2.5, 97.5], method="linear"; không làm tròn.
    """
    rng = np.random.default_rng(seed)
    medians = np.empty(n_boot, dtype=float)
    n = prices.size
    for i in range(n_boot):
        sample = rng.choice(prices, size=n, replace=True)
        medians[i] = np.median(sample)
    
    ci_low = float(np.percentile(medians, 2.5, method="linear"))
    ci_high = float(np.percentile(medians, 97.5, method="linear"))
    return {
        "medians": medians,
        "ci_low": ci_low,
        "ci_high": ci_high
    }


In [ ]:
def check_e2_1():
    prices = np.array([7.0])
    result = bootstrap_median(prices, n_boot=5, seed=42)
    assert isinstance(result, dict) and set(result) == {"medians", "ci_low", "ci_high"}
    assert isinstance(result["medians"], np.ndarray) and result["medians"].shape == (5,)
    assert np.issubdtype(result["medians"].dtype, np.floating)
    assert type(result["ci_low"]) is float and type(result["ci_high"]) is float
    np.testing.assert_allclose(result["medians"], [7.0] * 5)
    assert result["ci_low"] == 7.0 and result["ci_high"] == 7.0

public_check("E2 · mở rộng · ví dụ 1", check_e2_1)


[ĐẠT] E2 · mở rộng · ví dụ 1: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
def check_e2_2():
    prices = np.array([1.0, 2.0, 8.0])
    before = prices.copy()
    result = bootstrap_median(prices, n_boot=8, seed=17)
    repeat = bootstrap_median(prices, n_boot=8, seed=17)
    np.testing.assert_array_equal(result["medians"], repeat["medians"])
    # Đây là đối chiếu công khai cho một ví dụ nhỏ, không phải bộ hidden tests.
    expected = np.array([8.0, 2.0, 2.0, 2.0, 2.0, 2.0, 1.0, 2.0])
    np.testing.assert_allclose(result["medians"], expected)
    low, high = np.percentile(expected, [2.5, 97.5], method="linear")
    assert np.isclose(result["ci_low"], low) and np.isclose(result["ci_high"], high)
    np.testing.assert_array_equal(prices, before)

public_check("E2 · mở rộng · ví dụ 2", check_e2_2)


[ĐẠT] E2 · mở rộng · ví dụ 2: Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric.


In [ ]:
# Ví dụ nhỏ độc lập với Q1; không phải ước lượng cho Santiago.
preview("Bootstrap trên giá GIẢ LẬP", lambda: bootstrap_median(np.array([10.0, 20.0, 30.0]), n_boot=20, seed=42))

# Khi đã hoàn thành Q1 và E2, có thể chạy trên dữ liệu đang dùng:
# clean, _ = clean_prices(gia)
# result = bootstrap_median(clean, n_boot=1000, seed=42)


Bootstrap trên giá GIẢ LẬP
{'medians': array([20., 20., 10., 20., 30., 20., 20., 20., 20., 20., 10., 20., 30.,
       20., 20., 30., 30., 20., 20., 30.]), 'ci_low': 10.0, 'ci_high': 30.0}


### Giải thích phần mở rộng

- E1: so sánh trước/sau chuẩn hóa trên một hàng; hàng toàn 0 có thể nói gì về mùa vụ? 
    + Trước chuẩn hóa: Giá trị là số review tuyệt đối (bị ảnh hưởng lớn bởi quy mô tăng trưởng qua từng năm).
    + Sau chuẩn hóa: Mỗi tháng có hệ số so với trung bình năm đó. Giá trị $> 1.0$ thể hiện tháng cao điểm du lịch (ví dụ tháng 7, 8, 11), $< 1.0$ là tháng thấp điểm.
- E2: ghi nguồn dữ liệu, số vòng, seed, khoảng thu được và cách diễn giải.
    + Thực nghiệm với $n_boot = 1000$, $seed = 42$.
    + Khoảng tin cậy thu được $[ci_low, ci_high]$ cho biết khoảng biến thiên của trung vị mẫu dưới tác động của sự biến thiên lấy mẫu ngẫu nhiên từ dữ liệu quan sát.
- Vì sao tăng số lần bootstrap không tự sửa sai lệch do mẫu không đại diện? 
    + Bootstrap chỉ tái lấy mẫu (resample) từ chính tập dữ liệu mẫu hiện có. Nếu tập mẫu ban đầu đã bị chệch (sampling bias, thiếu đại diện cho toàn bộ thị trường), bootstrap chỉ mô phỏng độ bất định của chính mẫu lệch đó chứ không thể tạo thêm thông tin mới về phân phối thực tế của quần thể.
- Nếu chưa làm phần nâng cao, ghi “Chưa làm”; không tính thành lỗi các câu Q bắt buộc.


## Trước khi nộp

1. Điền tên, mã sinh viên và nguồn dữ liệu đã dùng ở cell dưới.
2. Restart & Run all; sửa các lỗi và đọc lại bảng public checks.
3. Hoàn thành câu trả lời diễn giải và khai báo AI, kể cả khi không dùng AI.
4. Giữ tên file và cell bài làm. Không chép kết quả hiển thị thành hằng số thay cho phép xử lý.
5. Lưu notebook vào repo bài nộp của bạn trong org, commit/push rồi tạo tag `submit/lab-03/v1` theo README.
   Chạy lại bằng tag v2 hoặc v3; không di chuyển tag đã nộp. **Pilot đã có private grader. Xem README của repo để tạo tag nộp bài và xem kết quả.**


**Họ tên / mã sinh viên:** Nguyễn Thành Vinh/25023073

**Nguồn dữ liệu thực hành:** [Giả lập để xem nháp / Santiago 2026-06-29; đường dẫn file]

**Khai báo AI cho phần được phép dùng:**

- Công cụ/model, hoặc “Không dùng”: Copilot
- Prompt chính: Giải thích các hàm như là argmin, mean, cách hoạt động của mask, tìm cú pháp.... Viết lại phần diễn đạt cho tôi (trình bày bằng lời để AI đổi sang filemd)
- Tôi đã tự dự đoán gì trước khi hỏi: AI sẽ cho tôi thông tin về những hàm pandas cần dùng
- Kết quả AI cần sửa hoặc điểm tôi chưa chắc: Không
- Cách tôi kiểm chứng, kèm ví dụ cụ thể: Viết thử, ví dụ như nó giải thích cách hoạt động của mask, tôi tạo thử mask = prices > 1 rồi kiểm tra lại df sau khi duyệt qua mask.



In [ ]:
show_public_summary()


PUBLIC CHECKS — không phải điểm chính thức
Q1 · ví dụ 1: ĐẠT
Q2 · ví dụ 1: ĐẠT
Q3 · ví dụ 1: ĐẠT
Q4 · ví dụ 1: ĐẠT
Q5 · ví dụ 1: ĐẠT
Q6 · ví dụ 1: ĐẠT
Q7 · ví dụ 1: ĐẠT
Q7 · ví dụ 2: ĐẠT
E1 · mở rộng · ví dụ 1: ĐẠT
E1 · mở rộng · ví dụ 2: ĐẠT
E2 · mở rộng · ví dụ 1: ĐẠT
E2 · mở rộng · ví dụ 2: ĐẠT
Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.
